In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
                    .appName("reading_invalid_events_dlq") \
                    .master("local[*]") \
                    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.1.2") \
                    .getOrCreate()

In [3]:
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, StringType

kafka_bootstrap_servers = "kafka1:19092,kafka2:19093,kafka3:19094"
dlq_topic = "events_dlq"

event_schema = StructType([
    StructField("event_id", LongType(), True), 
    StructField("user_id", IntegerType(), True),
    StructField("sequence_number", IntegerType(), True), 
    StructField("session_id", StringType(), True),
    StructField("created_at", StringType(), True), 
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),   
    StructField("browser", StringType(), True),
    StructField("traffic_source", StringType(), True), 
    StructField("uri", StringType(), True),
    StructField("event_type", StringType(), True),
])

Reading invalid events dlq

In [4]:
import pyspark.sql.functions as F

dlq = spark.read.format("kafka") \
       .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
       .option("subscribe", dlq_topic) \
       .option("startingOffsets", "earliest").option("endingOffsets", "latest") \
       .load() \
       .select(F.col("value").cast("string").alias("value")) \
       .dropDuplicates(["value"])

In [5]:
print(f"Invalid events in {dlq_topic} (unique): {dlq.count()}")

Invalid events in events_dlq (unique): 178


Add the reason why each events was invalidated

In [6]:
parsed = dlq.select("value", F.from_json("value", event_schema).alias("e"))

def broken(cond, message):
    """Return `message` when the rule is broken."""
    return F.when(F.coalesce(cond, F.lit(False)), F.lit(message))

created_ts = F.to_timestamp("e.created_at")
all_fields_null = F.coalesce(*[F.col(f"e.{f}").cast("string") for f in event_schema.fieldNames()]).isNull()

reason_list = F.concat_ws("; ",
    broken(all_fields_null, "not valid JSON"),
    broken(~all_fields_null & F.col("e.event_id").isNull(), "missing event_id"),
    broken(~all_fields_null & F.col("e.sequence_number").isNull(), "missing sequence_number"),
    broken(F.col("e.sequence_number") < 1, "sequence_number is 0 or negative"),
    broken(~all_fields_null & F.col("e.created_at").isNull(), "missing created_at"),
    broken(F.col("e.created_at").isNotNull() & created_ts.isNull(), "created_at is not a date"),
    broken(created_ts > F.current_timestamp() + F.expr("INTERVAL 1 MINUTE"), "created_at is in the future"),
)

reasons = parsed.withColumn(
    "reason",
    F.when(F.length(reason_list) == 0, F.lit("other")).otherwise(reason_list)
).select("value", "reason").cache()

In [7]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; } </style>"))

Reading invalid data

In [8]:
reasons.show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|value                                                                                                                                                                                                                                                                                                               |reason                          |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--

Summary of reasons

In [9]:
(reasons.select(F.explode(F.split("reason", "; ")).alias("reason"))
        .groupBy("reason").count().orderBy(F.desc("count"))
        .show(truncate=False))

+--------------------------------+-----+
|reason                          |count|
+--------------------------------+-----+
|sequence_number is 0 or negative|39   |
|created_at is in the future     |38   |
|created_at is not a date        |36   |
|not valid JSON                  |33   |
|missing event_id                |32   |
+--------------------------------+-----+

